In [1]:
# Cell 1: Codex suggested first cell

import torch, whisper

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FP16 = DEVICE == "cuda"
OUT_DIR = "transcription_outputs"

print("Device:", DEVICE)


Device: cuda


In [2]:
# Cell 2: Imports and configuration

from __future__ import annotations

import os
import shutil
from pathlib import Path
from pprint import pprint
from typing import Any


# ---- User-configurable paths ----
AUDIO_PATH = Path("doctor_patient_example2.wav")   # change this to your file
OUTPUT_DIR = Path("transcription_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TRANSCRIPT_TXT_PATH = OUTPUT_DIR / f"{AUDIO_PATH.stem}_transcript.txt"


def preview_dict(d: dict[str, Any], max_items: int = 8) -> dict[str, Any]:
    """
    Return a shallow preview of a dictionary for readable printing.
    Useful when Whisper returns nested objects and you want a quick look.
    """
    out = {}
    for i, (k, v) in enumerate(d.items()):
        if i >= max_items:
            break
        out[k] = v
    return out


print("Audio path:", AUDIO_PATH)
print("Output directory:", OUTPUT_DIR.resolve())
print("Transcript txt path:", TRANSCRIPT_TXT_PATH.resolve())

Audio path: doctor_patient_example2.wav
Output directory: /home/deuterium/whisper_app/transcription_outputs
Transcript txt path: /home/deuterium/whisper_app/transcription_outputs/doctor_patient_example2_transcript.txt


In [3]:
# Cell 3: Validate input path

def validate_audio_input(audio_path: Path) -> None:
    """
    Basic validation before attempting transcription.
    This keeps failures obvious and early.
    """
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Audio file not found: {audio_path}\n"
            "Update AUDIO_PATH to point to a real audio file."
        )

    if not audio_path.is_file():
        raise ValueError(f"Path is not a file: {audio_path}")

    if audio_path.stat().st_size == 0:
        raise ValueError(f"Audio file is empty: {audio_path}")

    if shutil.which("ffmpeg") is None:
        raise EnvironmentError(
            "ffmpeg was not found on your system PATH.\n"
            "Install ffmpeg first, then rerun the notebook.\n"
            "On Ubuntu, that is commonly: sudo apt-get install ffmpeg"
        )


validate_audio_input(AUDIO_PATH)
print("Input validation passed.")

Input validation passed.


In [4]:
# Cell 4: Load Whisper model (with simple CUDA OOM fallback)

import torch, whisper

MODEL_NAME = "medium.en"
print(f"Loading Whisper model: {MODEL_NAME} on {DEVICE}")

def load_with_fallback(name: str, device: str):
    if device == "cuda":
        try:
            torch.cuda.empty_cache()
            return whisper.load_model(name, device="cuda")
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print("CUDA OOM → falling back to CPU. Consider MODEL_NAME='small.en' to use GPU.")
                return whisper.load_model(name, device="cpu")
            raise
    return whisper.load_model(name, device=device)

model = load_with_fallback(MODEL_NAME, DEVICE)
print("Model ready.")


Loading Whisper model: medium.en on cuda
Model ready.


In [6]:
# Cell 5: Run transcription (no prompt)

from typing import Any

def transcribe_audio(model: Any, audio_path: Path) -> dict[str, Any]:
    return model.transcribe(
        str(audio_path),
        task="transcribe",
        language="en",
        beam_size=5,
        temperature=(0.0, 0.2, 0.4),  # fallback on tough segments
        condition_on_previous_text=True,  # keep context across windows
        fp16=False,
        verbose=False,
        word_timestamps=False,
    )

whisper_result = transcribe_audio(model, AUDIO_PATH)
print(whisper_result.get("text", "").strip()[:500])


100%|██████████| 20134/20134 [00:56<00:00, 355.48frames/s]

So, hello. My name is Daniela Gonzalez. I'm a second year medical student and this is my colleague, Napalia Flores, she's also a medical student. And today we're just gonna find out what brought you in here today. So, is your name Edith Riswell? Yes, it is. Yes, how would you like me to refer you? Edith is fine. Edith is fine? Alright. So, Edith, what brings you in here today? I'm just, I'm just feeling terrible and I'm losing too much weight for no reason. Oh, right. Can you tell me more? Yeah,


In [7]:
print(whisper_result.get("text", ""))

 So, hello. My name is Daniela Gonzalez. I'm a second year medical student and this is my colleague, Napalia Flores, she's also a medical student. And today we're just gonna find out what brought you in here today. So, is your name Edith Riswell? Yes, it is. Yes, how would you like me to refer you? Edith is fine. Edith is fine? Alright. So, Edith, what brings you in here today? I'm just, I'm just feeling terrible and I'm losing too much weight for no reason. Oh, right. Can you tell me more? Yeah, it's been probably, oh I don't know, maybe six or eight months that I've just been feeling more and more tired. It's gotten to a place where if I, no matter if I run to the store or, you know, just go do something, I'm, I'm warm right out and when I get home I just, I'm right down on the couch. So, I'm skipping extra stuff, social events and things like that. I just feel blah. I don't, there's no other way to put it. I just feel blah and I feel like something's wrong. And you told me that it's

In [8]:
print("Detected language:", whisper_result.get("language"))
segs = whisper_result.get("segments", [])[:5]
print("First 5 segment avg_logprob/compression_ratio:")
for s in segs:
    print(round(s.get("avg_logprob", 0), 3), "/", round(s.get("compression_ratio", 0), 3), "->", s.get("text","")[:80])


Detected language: en
First 5 segment avg_logprob/compression_ratio:
-0.352 / 1.606 ->  So, hello. My name is Daniela Gonzalez. I'm a second year medical student and t
-0.352 / 1.606 ->  And today we're just gonna find out what brought you in here today.
-0.352 / 1.606 ->  So, is your name Edith Riswell?
-0.352 / 1.606 ->  Yes, it is.
-0.352 / 1.606 ->  Yes, how would you like me to refer you?


In [9]:
# Cell 6: Inspect Whisper output structure

print("Type returned by model.transcribe(...):")
print(type(whisper_result))

print("\nTop-level keys:")
print(list(whisper_result.keys()))

print("\nTop-level preview:")
pprint(preview_dict(whisper_result))

segments = whisper_result.get("segments", [])
print("\nNumber of segments:")
print(len(segments))

if segments:
    first_segment = segments[0]
    print("\nType of first segment:")
    print(type(first_segment))

    print("\nKeys in first segment:")
    print(list(first_segment.keys()))

    print("\nFirst segment preview:")
    pprint(first_segment)

    print("\nReadable sample fields from first segment:")
    print("id:", first_segment.get("id"))
    print("start:", first_segment.get("start"))
    print("end:", first_segment.get("end"))
    print("text:", repr(first_segment.get("text")))
    print("avg_logprob:", first_segment.get("avg_logprob"))
    print("no_speech_prob:", first_segment.get("no_speech_prob"))
    print("compression_ratio:", first_segment.get("compression_ratio"))
else:
    print("No segments were returned.")

Type returned by model.transcribe(...):
<class 'dict'>

Top-level keys:
['text', 'segments', 'language']

Top-level preview:
{'language': 'en',
 'segments': [{'avg_logprob': -0.3524592798999232,
               'compression_ratio': 1.6059322033898304,
               'end': 12.32,
               'id': 0,
               'no_speech_prob': 0.01242303941398859,
               'seek': 0,
               'start': 0.0,
               'temperature': 0.0,
               'text': " So, hello. My name is Daniela Gonzalez. I'm a second "
                       'year medical student and this is my colleague, Napalia '
                       "Flores, she's also a medical student.",
               'tokens': [50363,
                          1406,
                          11,
                          23748,
                          13,
                          2011,
                          1438,
                          318,
                          7806,
                          64,
            

In [10]:
# Cell 7: Create intermediate pipeline schema

def build_transcript_segments(whisper_result: dict[str, Any]) -> list[dict[str, Any]]:
    """
    Standardize Whisper's segment output into a cleaner intermediate schema.

    This is the object you can hand to later pipeline stages such as:
    - diarization alignment
    - chunk merging
    - summarization
    - note generation
    - classification
    """
    transcript_segments: list[dict[str, Any]] = []

    for seg in whisper_result.get("segments", []):
        transcript_segments.append(
            {
                "segment_id": seg.get("id"),
                "start_sec": seg.get("start"),
                "end_sec": seg.get("end"),
                "duration_sec": (
                    seg.get("end") - seg.get("start")
                    if seg.get("start") is not None and seg.get("end") is not None
                    else None
                ),
                "text": (seg.get("text") or "").strip(),
                "tokens": seg.get("tokens"),  # token ids from Whisper, useful for debugging/advanced inspection
                "avg_logprob": seg.get("avg_logprob"),  # heuristic quality signal, not calibrated confidence
                "no_speech_prob": seg.get("no_speech_prob"),  # useful when filtering silence-like regions
                "compression_ratio": seg.get("compression_ratio"),  # useful for detecting odd decoding behavior
                "temperature": seg.get("temperature"),
                "source": "whisper"
            }
        )

    return transcript_segments


transcript_segments = build_transcript_segments(whisper_result)

# Higher-level object you can hand off to later stages
transcription_result = {
    "audio_path": str(AUDIO_PATH),
    "language": whisper_result.get("language"),
    "full_text": whisper_result.get("text", "").strip(),
    "transcript_segments": transcript_segments,
    "segment_count": len(transcript_segments),
    "engine": "openai-whisper",
    "model_name": MODEL_NAME,
}

pipeline_ready_transcript = transcription_result

print("Built intermediate schema: transcript_segments")
print("Number of standardized segments:", len(transcript_segments))

if transcript_segments:
    print("\nFirst standardized segment:")
    pprint(transcript_segments[0])

print("\nPipeline-ready top-level object keys:")
print(list(pipeline_ready_transcript.keys()))

Built intermediate schema: transcript_segments
Number of standardized segments: 33

First standardized segment:
{'avg_logprob': -0.3524592798999232,
 'compression_ratio': 1.6059322033898304,
 'duration_sec': 12.32,
 'end_sec': 12.32,
 'no_speech_prob': 0.01242303941398859,
 'segment_id': 0,
 'source': 'whisper',
 'start_sec': 0.0,
 'temperature': 0.0,
 'text': "So, hello. My name is Daniela Gonzalez. I'm a second year medical "
         "student and this is my colleague, Napalia Flores, she's also a "
         'medical student.',
 'tokens': [50363,
            1406,
            11,
            23748,
            13,
            2011,
            1438,
            318,
            7806,
            64,
            24416,
            13,
            314,
            1101,
            257,
            1218,
            614,
            3315,
            3710,
            290,
            428,
            318,
            616,
            16008,
            11,
            14332,
         

In [11]:
# Cell 8: Print the most pipeline-relevant fields

full_transcript_text = whisper_result.get("text", "").strip()

segment_level_text = [seg["text"] for seg in transcript_segments]
segment_times = [
    {
        "segment_id": seg["segment_id"],
        "start_sec": seg["start_sec"],
        "end_sec": seg["end_sec"]
    }
    for seg in transcript_segments
]
segment_quality_fields = [
    {
        "segment_id": seg["segment_id"],
        "avg_logprob": seg["avg_logprob"],
        "no_speech_prob": seg["no_speech_prob"],
        "compression_ratio": seg["compression_ratio"],
        "temperature": seg["temperature"]
    }
    for seg in transcript_segments
]

print("1. Full transcript text:")
print(full_transcript_text[:1000])

print("\n2. First 3 segment-level texts:")
for item in segment_level_text[:3]:
    print("-", item)

print("\n3. First 3 segment start/end times:")
for item in segment_times[:3]:
    print(item)

print("\n4. First 3 segment metadata / quality-related fields:")
for item in segment_quality_fields[:3]:
    print(item)

1. Full transcript text:
So, hello. My name is Daniela Gonzalez. I'm a second year medical student and this is my colleague, Napalia Flores, she's also a medical student. And today we're just gonna find out what brought you in here today. So, is your name Edith Riswell? Yes, it is. Yes, how would you like me to refer you? Edith is fine. Edith is fine? Alright. So, Edith, what brings you in here today? I'm just, I'm just feeling terrible and I'm losing too much weight for no reason. Oh, right. Can you tell me more? Yeah, it's been probably, oh I don't know, maybe six or eight months that I've just been feeling more and more tired. It's gotten to a place where if I, no matter if I run to the store or, you know, just go do something, I'm, I'm warm right out and when I get home I just, I'm right down on the couch. So, I'm skipping extra stuff, social events and things like that. I just feel blah. I don't, there's no other way to put it. I just feel blah and I feel like something's wrong. A

In [12]:
# Cell 9: Save transcript

def save_plain_transcript(text: str, output_path: Path) -> None:
    """
    Save plain transcript text to disk.
    """
    output_path.write_text(text, encoding="utf-8")


save_plain_transcript(full_transcript_text, TRANSCRIPT_TXT_PATH)

print(f"Saved plain transcript to: {TRANSCRIPT_TXT_PATH.resolve()}")

Saved plain transcript to: /home/deuterium/whisper_app/transcription_outputs/doctor_patient_example2_transcript.txt


In [13]:
# Cell 10: Optional sanity preview of the final handoff object

print("Pipeline-ready transcript summary:")
print({
    "audio_path": pipeline_ready_transcript["audio_path"],
    "language": pipeline_ready_transcript["language"],
    "segment_count": pipeline_ready_transcript["segment_count"],
    "engine": pipeline_ready_transcript["engine"],
    "model_name": pipeline_ready_transcript["model_name"],
})

print("\nFirst 2 standardized transcript segments:")
pprint(pipeline_ready_transcript["transcript_segments"][:2])

Pipeline-ready transcript summary:
{'audio_path': 'doctor_patient_example2.wav', 'language': 'en', 'segment_count': 33, 'engine': 'openai-whisper', 'model_name': 'medium.en'}

First 2 standardized transcript segments:
[{'avg_logprob': -0.3524592798999232,
  'compression_ratio': 1.6059322033898304,
  'duration_sec': 12.32,
  'end_sec': 12.32,
  'no_speech_prob': 0.01242303941398859,
  'segment_id': 0,
  'source': 'whisper',
  'start_sec': 0.0,
  'temperature': 0.0,
  'text': "So, hello. My name is Daniela Gonzalez. I'm a second year medical "
          "student and this is my colleague, Napalia Flores, she's also a "
          'medical student.',
  'tokens': [50363,
             1406,
             11,
             23748,
             13,
             2011,
             1438,
             318,
             7806,
             64,
             24416,
             13,
             314,
             1101,
             257,
             1218,
             614,
             3315,
             

In [14]:
# After whisper_result is available
import json

segments = [
    {"id": s["id"], "start": s["start"], "end": s["end"], "text": s["text"]}
    for s in whisper_result.get("segments", [])
]

structured = {
    "audio_path": str(AUDIO_PATH),
    "language": whisper_result.get("language"),
    "model": MODEL_NAME,
    "segments": segments,
}

json_path = OUTPUT_DIR / f"{AUDIO_PATH.stem}_segments.json"
json_path.write_text(json.dumps(structured, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", json_path)


Wrote: transcription_outputs/doctor_patient_example2_segments.json
